<a href="https://colab.research.google.com/github/Pes2ug23am092/AIS_transit_procurement_data_analysis/blob/main/Encephalon_ADA_2021_to_2023_prepro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Install required packages ---
!pip install geopandas shapely pyarrow --quiet

import pandas as pd
import numpy as np
from shapely import wkb, wkt
from math import radians, sin, cos, sqrt, atan2
from google.colab import drive

# --- Step 1: Mount Google Drive ---
drive.mount('/content/drive')

# --- Step 2: Load the parquet file ---
file_path = "/content/drive/MyDrive/LA_LB_AIS_2021_2023_CLEAN_FINAL.parquet"
df = pd.read_parquet(file_path)
print("✅ File loaded:", df.shape)

# --- Step 3: Safe WKB to geometry ---
def safe_wkb_to_geom(x):
    try:
        return wkb.loads(x)
    except Exception:
        return None

df["geometry"] = df["geometry"].apply(safe_wkb_to_geom)

# --- Step 4: Extract Lat/Lon ---
def extract_lat_lon(geom):
    if geom is None:
        return pd.Series([np.nan, np.nan])
    try:
        if geom.geom_type == "Point":
            return pd.Series([geom.y, geom.x])
        elif geom.geom_type == "LineString":
            y, x = geom.coords[0]
            return pd.Series([y, x])
        elif geom.geom_type == "MultiLineString":
            line = list(geom.geoms)[0]
            y, x = line.coords[0]
            return pd.Series([y, x])
        else:
            return pd.Series([np.nan, np.nan])
    except Exception:
        return pd.Series([np.nan, np.nan])

latlon = df["geometry"].apply(extract_lat_lon)
latlon.columns = ["LA", "LB"]
df = pd.concat([df, latlon], axis=1)
print("✅ Coordinates extracted:", df.shape)

# --- Step 5: Handle Missing / Invalid Data (impute) ---
# Draft_Meters: fill missing with median
median_draft = df['Draft_Meters'].median()
df['Draft_Meters'] = df['Draft_Meters'].fillna(median_draft)

# Coordinates: fill missing with median
df['LA'] = df['LA'].fillna(df['LA'].median())
df['LB'] = df['LB'].fillna(df['LB'].median())

# Flag invalid values
df['Draft_invalid'] = ~df['Draft_Meters'].between(0, 30)
df['LA_invalid'] = ~df['LA'].between(-90, 90)
df['LB_invalid'] = ~df['LB'].between(-180, 180)

# VesselType: fill missing and flag unknown
df['VesselType'] = df['VesselType'].fillna('unknown').astype(str)
df['VesselType_unknown'] = df['VesselType'].str.lower() == 'unknown'

# --- Step 6: Convert & Standardize Time ---
df['TrackStartTime'] = pd.to_datetime(df['TrackStartTime'], errors='coerce')
df['Year'] = df['TrackStartTime'].dt.year
df['Month'] = df['TrackStartTime'].dt.month
df['Day'] = df['TrackStartTime'].dt.day
df['Hour'] = df['TrackStartTime'].dt.hour
df['DayOfWeek'] = df['TrackStartTime'].dt.day_name()

# --- Step 7: Sort by vessel and time ---
df = df.sort_values(by=['MMSI','TrackStartTime']).reset_index(drop=True)

# --- Step 8: Compute Distance safely ---
def haversine(lat1, lon1, lat2, lon2):
    if np.any(pd.isna([lat1, lon1, lat2, lon2])):
        return 0  # replace missing distance with 0
    if not (-90 <= lat1 <= 90 and -90 <= lat2 <= 90 and -180 <= lon1 <= 180 and -180 <= lon2 <= 180):
        return 0
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

def compute_distance_safe(group):
    dist = [0]
    for i in range(1, len(group)):
        d = haversine(group['LA'].iloc[i-1], group['LB'].iloc[i-1],
                      group['LA'].iloc[i], group['LB'].iloc[i])
        dist.append(d)
    group['distance_km'] = dist
    return group

df = df.groupby('MMSI', group_keys=False).apply(compute_distance_safe).reset_index(drop=True)

# --- Step 9: Categorical Encoding ---
df['VesselTypeCode'] = df['VesselType'].astype('category').cat.codes

# --- Step 10: Feature Engineering ---
df['time_diff_hr'] = df.groupby('MMSI')['TrackStartTime'].diff().dt.total_seconds()/3600
df['time_diff_hr'] = df['time_diff_hr'].fillna(0.01)

df['avg_speed_kmph'] = df['distance_km'] / df['time_diff_hr']
df['avg_speed_kmph'] = df['avg_speed_kmph'].replace([np.inf,-np.inf],0).fillna(0)

df['abnormal_draft'] = df['Draft_Meters'] > 25
df['abnormal_speed'] = df['avg_speed_kmph'] > 80

# --- Step 11: Convert geometry to WKT for saving ---
df['geometry_wkt'] = df['geometry'].apply(lambda x: x.wkt if x else None)
df.drop(columns=['geometry'], inplace=True)

# --- Step 12: Missing Values Summary ---
missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isna().sum(),
    'Total_Rows': len(df)
})
missing_summary['Percent_Missing'] = missing_summary['Missing_Count'] / len(df) * 100

print("\n📊 Missing Values Summary (after imputation):")
print(missing_summary)

# --- Step 13: Save preprocessed dataset ---
df.to_parquet("/content/drive/MyDrive/LA_LB_AIS_2021_2023_PREPROCESSED.parquet", index=False)
print("\n✅ Preprocessed dataset saved to Drive. Final shape:", df.shape)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ File loaded: (143057, 5)
✅ Coordinates extracted: (143057, 7)


/tmp/ipython-input-2213538084.py:101: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('MMSI', group_keys=False).apply(compute_distance_safe).reset_index(drop=True)



📊 Missing Values Summary (after imputation):
                                Column  Missing_Count  Total_Rows  \
MMSI                              MMSI              0      143057   
TrackStartTime          TrackStartTime              0      143057   
VesselType                  VesselType              0      143057   
Draft_Meters              Draft_Meters              0      143057   
LA                                  LA              0      143057   
LB                                  LB              0      143057   
Draft_invalid            Draft_invalid              0      143057   
LA_invalid                  LA_invalid              0      143057   
LB_invalid                  LB_invalid              0      143057   
VesselType_unknown  VesselType_unknown              0      143057   
Year                              Year              0      143057   
Month                            Month              0      143057   
Day                                Day              0    

In [ ]:
from google.colab import files

# --- Path for saving ---
parquet_path = "/content/LA_LB_AIS_2021_2023_PREPROCESSED.parquet"

# --- Save as Parquet ---
df.to_parquet(parquet_path, index=False)
print("✅ Parquet file saved successfully!")

# --- Trigger download ---
files.download(parquet_path)


✅ Parquet file saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>